# 🔍 MTP 対応とフラグの静的確認（CPU）

vLLM 0.32 が `method=mtp` を受けるか、torchaudio 修復手順が効くかを先に確かめる。


In [ ]:
# MTP 対応の静的確認 + torchaudio 修復の実証（GPU 不要）
# 理由: MTP が vLLM 0.32 に無いなら、GPU ラダーの mtp 段は無駄。先に潰す。
import json
import os
import pathlib
import subprocess
import sys
import time

SPEC = "vllm==0.32.0"


def sh(cmd, timeout=3600):
    return subprocess.run(cmd, capture_output=True, text=True, timeout=timeout)


# install（dryrun と同じ手順）
t0 = time.time()
r = sh([sys.executable, "-c", "import vllm"])
if r.returncode != 0:
    print("pip install", SPEC)
    p = sh([sys.executable, "-m", "pip", "install", "-q", "--no-input", SPEC])
    print(p.returncode, ((p.stdout or "") + (p.stderr or ""))[-1200:])
print(f"install/確認: {time.time()-t0:.0f}s")


def cli_ok():
    r = sh(["vllm", "serve", "--help"], 300)
    out = (r.stdout or "") + (r.stderr or "")
    ok = (r.returncode == 0 and "Traceback" not in out and len(out) > 500)
    return ok, out


ok, out = cli_ok()
print("CLI OK:", ok, len(out), "文字")
fix = "none"
if not ok:
    print("⚠ traceback → torchaudio を外す")
    print(out[-800:])
    sh([sys.executable, "-m", "pip", "uninstall", "-y", "torchaudio"], 600)
    ok, out = cli_ok()
    fix = "uninstall"
print("CLI OK after:", ok, len(out), "文字 / fix:", fix)
if ok:
    open("/kaggle/working/vllm_help.txt", "w").write(out[:20000])

import vllm
print("vllm", vllm.__version__)
root = pathlib.Path(vllm.__file__).parent
print("root:", root)

# --- MTP の静的探索 ---
hits = []
for p in root.rglob("*.py"):
    try:
        txt = p.read_text(errors="replace")
    except Exception:
        continue
    low = txt.lower()
    if "mtp" not in low and "speculative" not in low:
        continue
    for i, line in enumerate(txt.splitlines(), 1):
        s = line.strip()
        if "mtp" in s.lower():
            hits.append(f"{p.relative_to(root)}:{i}: {s[:160]}")
print(f"mtp 出現: {len(hits)}")
print("\n".join(hits[:80]))
open("/kaggle/working/mtp_hits.txt", "w").write("\n".join(hits))

# --- SpeculativeConfig のスキーマ ---
report = {"cli_ok": ok, "fix": fix, "mtp_hits": len(hits)}
try:
    from vllm.config.speculative import SpeculativeConfig
    fields = getattr(SpeculativeConfig, "model_fields", None) or {}
    report["spec_fields"] = list(fields)
    print("fields:", list(fields))
    import inspect
    src = inspect.getsource(SpeculativeConfig)
    report["spec_src_head"] = src[:2500]
    print(src[:2500])
except Exception as e:
    report["spec_err"] = f"{type(e).__name__}: {e}"
    print("spec introspect NG:", report["spec_err"])

# --- 起動ラダーで使うフラグの実在確認 ---
NEED = ["--speculative-config", "--kv-cache-dtype", "--trust-remote-code",
        "--reasoning-parser", "--enable-auto-tool-choice", "--tool-call-parser",
        "--disable-log-requests", "--enforce-eager", "--max-num-seqs",
        "--served-model-name", "--tensor-parallel-size",
        "--gpu-memory-utilization", "--max-model-len", "--load-format"]
report["flags"] = {f: (f in out) for f in NEED}
print(json.dumps(report["flags"], indent=1))

with open("/kaggle/working/mtpprobe.json", "w") as f:
    json.dump(report, f, indent=2, ensure_ascii=False)
